# Übung: Process Mining mit dem Inductive Miner
## Datensatz: Hospital Billing – Krankenhausabrechnungsprozess

<!--
---

## Kontext

Der Datensatz stammt aus dem Abrechnungsmodul eines regionalen Krankenhauses. Jeder Trace entspricht einem **Billing Package** – einem Bündel medizinischer Leistungen, das abgerechnet werden soll. Der Log enthält 100.000 zufällig gezogene Prozessinstanzen aus drei Jahren Betrieb.

---

## Datensatz

**Datei:** `Hospital Billing - Event Log.xes.gz`  
**Aktivitäten (18 gesamt):**

| Aktivität      | Bedeutung                                        |
|----------------|--------------------------------------------------|
| `NEW`          | Neues Abrechnungspaket angelegt                  |
| `FIN`          | Finalisierung des Pakets                         |
| `RELEASE`      | Freigabe zur Abrechnung                          |
| `CODE OK`      | Diagnose-Code erfolgreich geprüft                |
| `BILLED`       | Rechnung erstellt und versendet                  |
| `CHANGE DIAGN` | Diagnose im Paket geändert                       |
| `DELETE`       | Paket gelöscht                                   |
| `REOPEN`       | Geschlossenes Paket wieder geöffnet              |
| `CODE NOK`     | Code-Prüfung fehlgeschlagen                      |
| `STORNO`       | Stornierung                                      |
| `REJECT`       | Ablehnung (z. B. durch Krankenkasse)             |
| weitere 7      | Seltene Sonderfälle (SET STATUS, MANUAL, …)      |

---

## Aufgaben

### Aufgabe 1 – Explorative Analyse

Laden Sie den Event Log und beantworten Sie folgende Fragen:

1. Wie viele Traces, Events und verschiedene Aktivitäten enthält der Log?
2. Wie lang sind die Traces im Durchschnitt und im Median?
3. Welche sind die **3 häufigsten Aktivitäten**?
4. Welche ist die **häufigste Variante** (Aktivitätenfolge) – und wie viel Prozent der Fälle deckt sie ab?
5. Wie viele verschiedene Varianten gibt es insgesamt?

Visualisieren Sie außerdem den **Directly-Follows-Graph (DFG)**.

---

### Aufgabe 2 – Prozessmodellierung mit dem Inductive Miner

Wenden Sie den **Inductive Miner – infrequent (IMf)** mit zwei verschiedenen Rauschschwellen an:

| Parameterwert           | Bedeutung                                          |
|-------------------------|----------------------------------------------------|
| `noise_threshold = 0.2` | Seltene Pfade werden kaum herausgefiltert          |
| `noise_threshold = 0.5` | Häufige Abweichungen werden als Rauschen ignoriert |

1. Entdecken Sie für **beide** Schwellenwerte jeweils einen **Prozessbaum**.
2. Visualisieren Sie beide Bäume und beschreiben Sie kurz den strukturellen Unterschied.
3. Welche Operatoren erkennen Sie im Baum? Erklären Sie deren Bedeutung an einem konkreten Beispiel aus dem Modell.

---

### Aufgabe 3 – Conformance Checking

Wenden Sie **Token-based Replay** auf **beide Modelle** an und berechnen Sie die **Fitness**.

1. Tragen Sie die Ergebnisse in folgende Tabelle ein:

| Modell                   | Fitness |
|--------------------------|---------|
| IMf, noise_threshold=0.2 |         |
| IMf, noise_threshold=0.5 |         |

2. Erklären Sie: Warum **sinkt** die Fitness, wenn `noise_threshold` steigt?
3. Für welches Modell würden Sie sich entscheiden, wenn Sie das Ergebnis einem Krankenhausmanager präsentieren möchten? Begründen Sie kurz.

-->

In [ ]:
# ============================================================
# Lösung: Inductive Miner – Hospital Billing Event Log
# Voraussetzung: pip install pm4py
# ============================================================


# ─────────────────────────────────────────────
# Preliminaries: Load data
# ─────────────────────────────────────────────

# %pip install pm4py

import pm4py
import warnings
warnings.filterwarnings('ignore')

# Log laden
#log = pm4py.read_xes("/content/Hospital Billing - Event Log.xes.gz")
#log = pm4py.read_xes("/content/drive/MyDrive/Colab Notebooks/Learning Labs Processmanagement/Hospital Billing - Event Log.xes.gz")
log = pm4py.read_xes("Hospital Billing - Event Log.xes.gz")

# DataFrame für einfache Auswertung
df = pm4py.convert_to_dataframe(log)

In [ ]:
# ─────────────────────────────────────────────
# AUFGABE 1: Explorative Analyse
# ─────────────────────────────────────────────

# Log laden
#log = pm4py.read_xes("Hospital Billing - Event Log.xes.gz")

# DataFrame für einfache Auswertung
df = pm4py.convert_to_dataframe(log)

# Basiskennzahlen
n_traces     = df['case:concept:name'].nunique()
n_events     = len(df)
n_activities = df['concept:name'].nunique()
trace_len    = df.groupby('case:concept:name').size()

print(f"Traces:                   {n_traces:,}")
print(f"Events:                   {n_events:,}")
print(f"Verschiedene Aktivitäten: {n_activities}")
print(f"Ø Trace-Länge:            {trace_len.mean():.2f}")
print(f"Median Trace-Länge:       {trace_len.median():.0f}")

# Top-3 Aktivitäten
print("\nTop-3 Aktivitäten:")
print(df['concept:name'].value_counts().head(3))

# Varianten
variants = pm4py.get_variants(log)
sorted_v  = sorted(variants.items(), key=lambda x: x[1], reverse=True)

print(f"\nAnzahl Varianten: {len(variants)}")

top_v, top_count = sorted_v[0]
print(f"Häufigste Variante ({top_count:,} Fälle, "
      f"{top_count / n_traces * 100:.1f}%): {' → '.join(top_v)}")

# DFG visualisieren
dfg, start, end = pm4py.discover_dfg(log)
pm4py.view_dfg(dfg, start, end)

In [ ]:
# ─────────────────────────────────────────────
# AUFGABE 2: Inductive Miner – Prozessbäume
# ─────────────────────────────────────────────


# Prozessbaum ohne noise_threshold
tree_00 = pm4py.discover_process_tree_inductive(log)
pm4py.view_process_tree(tree_00)

tree_bpmn = pm4py.discover_bpmn_inductive(log)
pm4py.view_bpmn(tree_bpmn)

# Prozessbaum mit noise_threshold = 0.2
tree_02 = pm4py.discover_process_tree_inductive(log, noise_threshold=0.2)
pm4py.view_process_tree(tree_02)

tree_bpmn_02 = pm4py.discover_bpmn_inductive(log, noise_threshold=0.2)
pm4py.view_bpmn(tree_bpmn_02)

# Prozessbaum mit noise_threshold = 0.5
tree_05 = pm4py.discover_process_tree_inductive(log, noise_threshold=0.5)
pm4py.view_process_tree(tree_05)

tree_bpmn_05 = pm4py.discover_bpmn_inductive(log, noise_threshold=0.5)
pm4py.view_bpmn(tree_bpmn_05)



In [ ]:
# ─────────────────────────────────────────────
# AUFGABE 3: Conformance Checking – Fitness
# ─────────────────────────────────────────────

from pm4py.algo.conformance.tokenreplay import algorithm as token_replay

# Modelle als Petri-Netz (intern für Token Replay benötigt)
net_02, im_02, fm_02 = pm4py.discover_petri_net_inductive(log, noise_threshold=0.2)
net_05, im_05, fm_05 = pm4py.discover_petri_net_inductive(log, noise_threshold=0.5)

# Token-based Replay
tbr_02 = token_replay.apply(log, net_02, im_02, fm_02)
tbr_05 = token_replay.apply(log, net_05, im_05, fm_05)

fitness_02 = sum(t['trace_fitness'] for t in tbr_02) / len(tbr_02)
fitness_05 = sum(t['trace_fitness'] for t in tbr_05) / len(tbr_05)

print(f"Fitness noise=0.2: {fitness_02:.4f}")  # → 0.9854
print(f"Fitness noise=0.5: {fitness_05:.4f}")  # → 0.9680